In [0]:
from pyspark.sql.functions import *

catalog = "bike_data"
bronze_schema = "bronze"
silver_schema = "silver"

print("=" * 80)
print("SILVER TRANSFORMATION: erp_cust_az12")
print("=" * 80)

# Section 1: Read Bronze table
print("\nSection 1: Reading Bronze table")
df = spark.table(f"{catalog}.{bronze_schema}.erp_cust_az12")
initial_count = df.count()
print(f"Initial rows: {initial_count}")

# Section 2: Transform data
print("\nSection 2: Data transformation")

# 2.1 Remove duplicates
print("  2.1 Removing duplicates by CID")
df = df.dropDuplicates(subset=["CID"])
print(f"  Rows after dedup: {df.count()}")

# 2.2 Clean strings
print("  2.2 Cleaning string columns")
df = df.withColumn("GEN", trim(upper(col("GEN"))))

# 2.3 Fix dates
print("  2.3 Fixing date columns")
df = df.withColumn("BDATE",
    coalesce(
        to_date(col("BDATE"), "yyyy-MM-dd"),
        to_date(col("BDATE"), "MM/dd/yyyy"),
        to_date(col("BDATE"), "dd-MM-yyyy")
    )
)

# 2.4 Remove invalid rows
print("  2.4 Removing invalid rows")
df = df.filter(col("CID").isNotNull())

print(f"  Rows after transformations: {df.count()}")

# Section 3: Sanity checks
print("\nSection 3: Sanity checks")
null_check = df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns])
display(null_check)
display(df.limit(3))

# Section 4: Write to Silver
print("\nSection 4: Writing to Silver table")
silver_table = f"{catalog}.{silver_schema}.erp_customers"
df.write.mode("overwrite").format("delta").saveAsTable(silver_table)

final_count = df.count()
print(f"Written to: {silver_table}")
print(f"Final rows: {final_count}")
print(f"Removed: {initial_count - final_count} rows")